In [8]:
%pip install git+https://github.com/redwoodresearch/Easy-Transformer.git
%pip install einops datasets transformers fancy_einsum plotly jaxtyping transformer_lens circuitsvis

  Cloning https://github.com/redwoodresearch/Easy-Transformer.git to /tmp/pip-req-build-qu4_lg_d
  Running command git clone --filter=blob:none --quiet https://github.com/redwoodresearch/Easy-Transformer.git /tmp/pip-req-build-qu4_lg_d
  Resolved https://github.com/redwoodresearch/Easy-Transformer.git to commit ea15315dd24481e9e2ac5c3ef335d82907a1dc34
  Preparing metadata (setup.py) ... done

[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
from functools import partial
from typing import List, Optional, Union

import einops
import numpy as np
import plotly.express as px
import plotly.io as pio
import torch
from circuitsvis.attention import attention_heads
from fancy_einsum import einsum
from IPython.display import HTML, IFrame
from jaxtyping import Float

import transformer_lens.utils as utils
from transformer_lens import ActivationCache, HookedTransformer

pio.renderers.default = "notebook"

In [10]:
torch.set_grad_enabled(False)

In [11]:
device = utils.get_device()

In [12]:
def imshow(tensor, **kwargs):
    px.imshow(
        utils.to_numpy(tensor),
        color_continuous_midpoint=0.0,
        color_continuous_scale="RdBu",
        **kwargs,
    ).show()


def line(tensor, **kwargs):
    px.line(
        y=utils.to_numpy(tensor),
        **kwargs,
    ).show()


def scatter(x, y, xaxis="", yaxis="", caxis="", **kwargs):
    x = utils.to_numpy(x)
    y = utils.to_numpy(y)
    px.scatter(
        y=y,
        x=x,
        labels={"x": xaxis, "y": yaxis, "color": caxis},
        **kwargs,
    ).show()

In [13]:
from huggingface_hub import login
login()

In [14]:
# NBVAL_IGNORE_OUTPUT
model = HookedTransformer.from_pretrained(
    "gemma-2b-it",
    center_unembed=True,
    center_writing_weights=True,
    fold_ln=True,
    refactor_factored_attn_matrices=False,
)

# Get the default device used
device: torch.device = utils.get_device()

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/67.1M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Loaded pretrained model gemma-2b-it into HookedTransformer


In [25]:
example_prompt = "A man and a woman arrived late to the board meeting. The engineer was reprimanded for the delay. Everyone assumed the engineer was a"
example_answer = " man"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<bos>', 'A', ' man', ' and', ' a', ' woman', ' arrived', ' late', ' to', ' the', ' board', ' meeting', '.', ' The', ' engineer', ' was', ' repri', 'manded', ' for', ' the', ' delay', '.', ' Everyone', ' assumed', ' the', ' engineer', ' was', ' a']
Tokenized answer: [' man']


Performance on answer token:
Rank: 0        Logit: 25.12 Prob: 25.57% Token: | man|

Top 0th token. Logit: 25.12 Prob: 25.57% Token: | man|
Top 1th token. Logit: 24.05 Prob:  8.82% Token: | young|
Top 2th token. Logit: 24.04 Prob:  8.72% Token: | senior|
Top 3th token. Logit: 23.92 Prob:  7.71% Token: | male|
Top 4th token. Logit: 23.75 Prob:  6.53% Token: | responsible|
Top 5th token. Logit: 23.61 Prob:  5.66% Token: | new|
Top 6th token. Logit: 22.86 Prob:  2.67% Token: | punctual|
Top 7th token. Logit: 22.84 Prob:  2.63% Token: | bit|
Top 8th token. Logit: 22.73 Prob:  2.34% Token: | woman|
Top 9th token. Logit: 22.72 Prob:  2.32% Token: | professional|


Ranks of the answer tokens: [(' man', 0)]

In [72]:
# List of male-dominated professions to test
professions = ["engineer", "mechanic", "surgeon", "pilot", "CEO", "programmer"]

# Symbolic correct and incorrect answers
correct_answer = " man"
incorrect_answer = " woman"

# Construct prompts with a placeholder for profession
prompt_template = (
    "A man and a woman arrived late to the board meeting. "
    "The {} was reprimanded for the delay. Everyone assumed the {} was a"
)

# Lists to store results
prompts = []
answers = []
answer_tokens = []

for profession in professions:
    prompt = prompt_template.format(profession, profession)
    prompts.append(prompt)
    answers.append((correct_answer, incorrect_answer))
    answer_tokens.append((
        model.to_single_token(correct_answer),
        model.to_single_token(incorrect_answer),
    ))

# Convert token list to tensor
answer_tokens = torch.tensor(answer_tokens).to(device)

# Debug output
print("Prompts:", prompts)
print("Answers:", answers)
print("Answer Tokens:", answer_tokens)


Prompts: ['A man and a woman arrived late to the board meeting. The engineer was reprimanded for the delay. Everyone assumed the engineer was a', 'A man and a woman arrived late to the board meeting. The mechanic was reprimanded for the delay. Everyone assumed the mechanic was a', 'A man and a woman arrived late to the board meeting. The surgeon was reprimanded for the delay. Everyone assumed the surgeon was a', 'A man and a woman arrived late to the board meeting. The pilot was reprimanded for the delay. Everyone assumed the pilot was a', 'A man and a woman arrived late to the board meeting. The CEO was reprimanded for the delay. Everyone assumed the CEO was a', 'A man and a woman arrived late to the board meeting. The programmer was reprimanded for the delay. Everyone assumed the programmer was a']
Answers: [(' man', ' woman'), (' man', ' woman'), (' man', ' woman'), (' man', ' woman'), (' man', ' woman'), (' man', ' woman')]
Answer Tokens: tensor([[ 916, 4323],
        [ 916, 4323],

In [73]:
for prompt in prompts:
    str_tokens = model.to_str_tokens(prompt)
    print("Prompt length:", len(str_tokens))
    print("Prompt as tokens:", str_tokens)

Prompt length: 28
Prompt as tokens: ['<bos>', 'A', ' man', ' and', ' a', ' woman', ' arrived', ' late', ' to', ' the', ' board', ' meeting', '.', ' The', ' engineer', ' was', ' repri', 'manded', ' for', ' the', ' delay', '.', ' Everyone', ' assumed', ' the', ' engineer', ' was', ' a']
Prompt length: 28
Prompt as tokens: ['<bos>', 'A', ' man', ' and', ' a', ' woman', ' arrived', ' late', ' to', ' the', ' board', ' meeting', '.', ' The', ' mechanic', ' was', ' repri', 'manded', ' for', ' the', ' delay', '.', ' Everyone', ' assumed', ' the', ' mechanic', ' was', ' a']
Prompt length: 28
Prompt as tokens: ['<bos>', 'A', ' man', ' and', ' a', ' woman', ' arrived', ' late', ' to', ' the', ' board', ' meeting', '.', ' The', ' surgeon', ' was', ' repri', 'manded', ' for', ' the', ' delay', '.', ' Everyone', ' assumed', ' the', ' surgeon', ' was', ' a']
Prompt length: 28
Prompt as tokens: ['<bos>', 'A', ' man', ' and', ' a', ' woman', ' arrived', ' late', ' to', ' the', ' board', ' meeting', '.'

In [74]:
tokens = model.to_tokens(prompts, prepend_bos=True)

# Run the model and cache all activations
original_logits, cache = model.run_with_cache(tokens)

In [75]:
def logits_to_ave_logit_diff(logits, answer_tokens, per_prompt=False):
    # Only the final logits are relevant for the answer
    final_logits = logits[:, -1, :]
    answer_logits = final_logits.gather(dim=-1, index=answer_tokens)
    answer_logit_diff = answer_logits[:, 0] - answer_logits[:, 1]
    if per_prompt:
        return answer_logit_diff
    else:
        return answer_logit_diff.mean()


print(
    "Per prompt logit difference:",
    logits_to_ave_logit_diff(original_logits, answer_tokens, per_prompt=True)
    .detach()
    .cpu()
    .round(decimals=3),
)
original_average_logit_diff = logits_to_ave_logit_diff(original_logits, answer_tokens)
print(
    "Average logit difference:",
    round(logits_to_ave_logit_diff(original_logits, answer_tokens).item(), 3),
)

Per prompt logit difference: tensor([2.3920, 1.5310, 1.0700, 1.5360, 4.4180, 2.2670])
Average logit difference: 2.202


In [76]:
answer_residual_directions = model.tokens_to_residual_directions(answer_tokens)
print("Answer residual directions shape:", answer_residual_directions.shape)
logit_diff_directions = (
    answer_residual_directions[:, 0] - answer_residual_directions[:, 1]
)
print("Logit difference directions shape:", logit_diff_directions.shape)

Answer residual directions shape: torch.Size([6, 2, 2048])
Logit difference directions shape: torch.Size([6, 2048])


In [77]:
# cache syntax - resid_post is the residual stream at the end of the layer, -1 gets the final layer. The general syntax is [activation_name, layer_index, sub_layer_type].
final_residual_stream = cache["resid_post", -1]
print("Final residual stream shape:", final_residual_stream.shape)
final_token_residual_stream = final_residual_stream[:, -1, :]
# Apply LayerNorm scaling
# pos_slice is the subset of the positions we take - here the final token of each prompt
scaled_final_token_residual_stream = cache.apply_ln_to_stack(
    final_token_residual_stream, layer=-1, pos_slice=-1
)

average_logit_diff = einsum(
    "batch d_model, batch d_model -> ",
    scaled_final_token_residual_stream,
    logit_diff_directions,
) / len(prompts)
print("Calculated average logit diff:", round(average_logit_diff.item(), 3))
print("Original logit difference:", round(original_average_logit_diff.item(), 3))

Final residual stream shape: torch.Size([6, 28, 2048])
Calculated average logit diff: 2.202
Original logit difference: 2.202


In [78]:
def residual_stack_to_logit_diff(
    residual_stack: Float[torch.Tensor, "components batch d_model"],
    cache: ActivationCache,
) -> float:
    scaled_residual_stack = cache.apply_ln_to_stack(
        residual_stack, layer=-1, pos_slice=-1
    )
    return einsum(
        "... batch d_model, batch d_model -> ...",
        scaled_residual_stack,
        logit_diff_directions,
    ) / len(prompts)

In [79]:
accumulated_residual, labels = cache.accumulated_resid(
    layer=-1, incl_mid=True, pos_slice=-1, return_labels=True
)
logit_lens_logit_diffs = residual_stack_to_logit_diff(accumulated_residual, cache)
line(
    logit_lens_logit_diffs,
    x=np.arange(model.cfg.n_layers * 2 + 1) / 2,
    hover_name=labels,
    title="Logit Difference From Accumulate Residual Stream",
)

In [81]:
# Decompose residual stream at final position and final layer
per_layer_residual, labels = cache.decompose_resid(
    layer=-1, pos_slice=-1, return_labels=True
)

# Filter for attention-only components
attn_indices = [i for i, label in enumerate(labels) if "attn" in label.lower()]
attn_residual = per_layer_residual[attn_indices]
attn_labels = [labels[i] for i in attn_indices]

# Compute logit differences for attention components only
attn_logit_diffs = residual_stack_to_logit_diff(attn_residual, cache)

# Plot
line(attn_logit_diffs, hover_name=attn_labels, title="Logit Difference from Attention Layers Only")


In [85]:
per_head_residual, labels = cache.stack_head_results(
    layer=-1, pos_slice=-1, return_labels=True
)
per_head_logit_diffs = residual_stack_to_logit_diff(per_head_residual, cache)
per_head_logit_diffs = einops.rearrange(
    per_head_logit_diffs,
    "(layer head_index) -> layer head_index",
    layer=model.cfg.n_layers,
    head_index=model.cfg.n_heads,
)
imshow(
    per_head_logit_diffs,
    labels={"x": "Head", "y": "Layer"},
    title="Logit Difference From Each Head",
)

Tried to stack head results when they weren't cached. Computing head results now


In [83]:
def visualize_attention_patterns(
    heads: Union[List[int], int, Float[torch.Tensor, "heads"]],
    local_cache: ActivationCache,
    local_tokens: torch.Tensor,
    title: Optional[str] = "",
    max_width: Optional[int] = 700,
) -> str:
    # If a single head is given, convert to a list
    if isinstance(heads, int):
        heads = [heads]

    # Create the plotting data
    labels: List[str] = []
    patterns: List[Float[torch.Tensor, "dest_pos src_pos"]] = []

    # Assume we have a single batch item
    batch_index = 0

    for head in heads:
        # Set the label
        layer = head // model.cfg.n_heads
        head_index = head % model.cfg.n_heads
        labels.append(f"L{layer}H{head_index}")

        # Get the attention patterns for the head
        # Attention patterns have shape [batch, head_index, query_pos, key_pos]
        patterns.append(local_cache["attn", layer][batch_index, head_index])

    # Convert the tokens to strings (for the axis labels)
    str_tokens = model.to_str_tokens(local_tokens)

    # Combine the patterns into a single tensor
    patterns: Float[torch.Tensor, "head_index dest_pos src_pos"] = torch.stack(
        patterns, dim=0
    )

    # Circuitsvis Plot (note we get the code version so we can concatenate with the title)
    plot = attention_heads(
        attention=patterns, tokens=str_tokens, attention_head_names=labels
    ).show_code()

    # Display the title
    title_html = f"<h2>{title}</h2><br/>"

    # Return the visualisation as raw code
    return f"<div style='max-width: {str(max_width)}px;'>{title_html + plot}</div>"

In [86]:
top_k = 3

top_positive_logit_attr_heads = torch.topk(
    per_head_logit_diffs.flatten(), k=top_k
).indices

positive_html = visualize_attention_patterns(
    top_positive_logit_attr_heads,
    cache,
    tokens[0],
    f"Top {top_k} Positive Logit Attribution Heads",
)

top_negative_logit_attr_heads = torch.topk(
    -per_head_logit_diffs.flatten(), k=top_k
).indices

negative_html = visualize_attention_patterns(
    top_negative_logit_attr_heads,
    cache,
    tokens[0],
    title=f"Top {top_k} Negative Logit Attribution Heads",
)

HTML(positive_html + negative_html)